# Actividad 2 — Principio de Abierto/Cerrado (OCP)

**Estudiante:** Andrés Felipe Luna Camargo  
**Dominio:** Liquidación y Tarifas de Almacenamiento en Bodegas Frigoríficas

---

## 1. Ejemplo Incorrecto (Violando OCP)

En este ejemplo tenemos una clase `LiquidadorTarifas` con un método que usa una serie de `if/elif/else` para calcular cuánto cobrarle a un cliente según el tipo de cuarto que alquiló:
- `"refrigerado"`: frutas o verduras a +4°C.
- `"congelado"`: carnes o pescados a -18°C.
- `"ultracongelado"`: vacunas o biológicos a -80°C.



In [ ]:
# Diseño rígido con if/elif
class LiquidadorTarifasRigido:
    def __init__(self, nombre_bodega: str, descuento_mayorista: float = 0.05) -> None:
        self.nombre_bodega: str = nombre_bodega
        self.descuento_mayorista: float = descuento_mayorista

    def calcular_cobro(self, tipo_camara: str, pallets: int, dias: int) -> float:
        # Cadena de if/elif que toca modificar cada vez que inventemos un servicio nuevo
        tipo = tipo_camara.lower()
        if tipo == "refrigerado":
            tarifa_pallet_dia = 5000.0
            factor_energia = 1.0
        elif tipo == "congelado":
            tarifa_pallet_dia = 12000.0
            factor_energia = 1.3
        elif tipo == "ultracongelado":
            tarifa_pallet_dia = 35000.0
            factor_energia = 1.8
        else:
            raise ValueError(f"Tipo de servicio '{tipo_camara}' no existe en el sistema.")

        subtotal = pallets * dias * tarifa_pallet_dia * factor_energia
        if pallets >= 50:
            subtotal -= subtotal * self.descuento_mayorista
        return round(subtotal, 2)

    def generar_recibo(self, cliente: str, tipo_camara: str, pallets: int, dias: int) -> str:
        total = self.calcular_cobro(tipo_camara, pallets, dias)
        return f"[{self.nombre_bodega}] Cliente: {cliente} | Servicio: {tipo_camara} | Total: ${total:,.2f} COP"


Clase LiquidadorTarifasRigido creada.


In [2]:
# Probamos la versión rígida
print("--- Probando Liquidador Rígido (Sin OCP) ---")
liquidador = LiquidadorTarifasRigido("Frigoríficos del Norte", descuento_mayorista=0.10)

print(liquidador.generar_recibo("AgroFrutas S.A.", "refrigerado", pallets=30, dias=15))
print(liquidador.generar_recibo("Avícola Central", "congelado", pallets=60, dias=30))
print(liquidador.generar_recibo("Laboratorio Farma", "ultracongelado", pallets=10, dias=20))

# Si llega un cliente pidiendo criogenia con nitrógeno:
try:
    print(liquidador.generar_recibo("BioBanco Células", "criogenia", pallets=5, dias=30))
except ValueError as err:
    print(f"\nError esperado: {err}")
    print("-> Para arreglarlo nos toca entrar a editar el código de LiquidadorTarifasRigido.")


--- Probando Liquidador Rígido (Sin OCP) ---
[Frigoríficos del Norte] Cliente: AgroFrutas S.A. | Servicio: refrigerado | Total: $2,250,000.00 COP
[Frigoríficos del Norte] Cliente: Avícola Central | Servicio: congelado | Total: $25,272,000.00 COP
[Frigoríficos del Norte] Cliente: Laboratorio Farma | Servicio: ultracongelado | Total: $12,600,000.00 COP

Error esperado: Tipo de servicio 'criogenia' no existe en el sistema.
-> Para arreglarlo nos toca entrar a editar el código de LiquidadorTarifasRigido.


## 2. Ejemplo Correcto (Aplicando OCP)

Para resolverlo usamos **polimorfismo y clases abstractas**:
1. Creamos una clase base abstracta `TarifaServicio(ABC)` que define el método `calcular_cobro(pallets, dias)`.
2. Cada tipo de almacenamiento se convierte en su propia clase:
   - `TarifaRefrigerado`
   - `TarifaCongelado`
   - `TarifaUltraCongelado`
3. La clase `FacturadorBodega` recibe cualquier objeto que herede de `TarifaServicio` y calcula el total sin usar un solo `if/elif`.
4. Si mañana queremos agregar un nuevo servicio (como `TarifaCriogenica`), simplemente creamos esa nueva clase **sin tocar ni una sola línea del facturador**.


In [ ]:
from abc import ABC, abstractmethod

#  Clase base abstracta
class TarifaServicio(ABC):
    @abstractmethod
    def calcular_costo_base(self, pallets: int, dias: int) -> float:
        pass

    @abstractmethod
    def obtener_nombre_servicio(self) -> str:
        pass


# Implementaciones concretas existentes
class TarifaRefrigerado(TarifaServicio):
    def __init__(self, precio_pallet_dia: float = 5000.0, recargo_humedad: float = 500.0) -> None:
        self.precio_pallet_dia: float = precio_pallet_dia
        self.recargo_humedad: float = recargo_humedad

    def calcular_costo_base(self, pallets: int, dias: int) -> float:
        costo_unitario = self.precio_pallet_dia + self.recargo_humedad
        return round(pallets * dias * costo_unitario, 2)

    def obtener_nombre_servicio(self) -> str:
        return "Refrigeración Frescos (+4°C con control de humedad)"


class TarifaCongelado(TarifaServicio):
    def __init__(self, precio_pallet_dia: float = 12000.0, factor_frio: float = 1.3) -> None:
        self.precio_pallet_dia: float = precio_pallet_dia
        self.factor_frio: float = factor_frio

    def calcular_costo_base(self, pallets: int, dias: int) -> float:
        return round(pallets * dias * self.precio_pallet_dia * self.factor_frio, 2)

    def obtener_nombre_servicio(self) -> str:
        return "Congelados Estándar (-18°C)"


class TarifaUltraCongelado(TarifaServicio):
    def __init__(self, precio_pallet_dia: float = 35000.0, cargo_planta_electrica: float = 200000.0) -> None:
        self.precio_pallet_dia: float = precio_pallet_dia
        self.cargo_planta_electrica: float = cargo_planta_electrica

    def calcular_costo_base(self, pallets: int, dias: int) -> float:
        costo_espacio = pallets * dias * self.precio_pallet_dia * 1.5
        return round(costo_espacio + self.cargo_planta_electrica, 2)

    def obtener_nombre_servicio(self) -> str:
        return "Ultra-Freezer Farmacéutico (-80°C con generador de respaldo)"


# Facturador cerrado a modificación (usa polimorfismo)
class FacturadorBodega:
    def __init__(self, nombre_empresa: str, descuento_por_volumen: float = 0.10) -> None:
        self.nombre_empresa: str = nombre_empresa
        self.descuento_por_volumen: float = descuento_por_volumen

    def liquidar(self, tarifa: TarifaServicio, pallets: int, dias: int) -> dict:
        subtotal = tarifa.calcular_costo_base(pallets, dias)
        descuento = 0.0
        if pallets >= 50:
            descuento = subtotal * self.descuento_por_volumen
        total = subtotal - descuento
        return {
            "servicio": tarifa.obtener_nombre_servicio(),
            "subtotal": round(subtotal, 2),
            "descuento": round(descuento, 2),
            "total": round(total, 2)
        }

    def imprimir_cuenta(self, cliente: str, tarifa: TarifaServicio, pallets: int, dias: int) -> None:
        resultado = self.liquidar(tarifa, pallets, dias)
        print(f"Factura: {self.nombre_empresa}")
        print(f"Cliente: {cliente}")
        print(f"Tipo: {resultado['servicio']}")
        print(f"Cantidad: {pallets} pallets por {dias} días")
        print(f"Subtotal: ${resultado['subtotal']:,.2f} | Descuento: ${resultado['descuento']:,.2f} | Total: ${resultado['total']:,.2f} COP")
        print("-" * 70)


Clases modulares con OCP creadas correctamente.


In [ ]:
# Demostración de EXTENSIÓN Agregamos Criogenia sin tocar FacturadorBodega

# Creamos la nueva tarifa simplemente heredando de TarifaServicio
class TarifaCriogenica(TarifaServicio):
    def __init__(self, precio_m3_dia: float = 65000.0, costo_recarga_n2: float = 150000.0) -> None:
        self.precio_m3_dia: float = precio_m3_dia
        self.costo_recarga_n2: float = costo_recarga_n2

    def calcular_costo_base(self, pallets: int, dias: int) -> float:
        # En criogenia se cobra por pallet especial y recargas de nitrógeno líquido
        costo_almacen = pallets * dias * self.precio_m3_dia
        return round(costo_almacen + self.costo_recarga_n2, 2)

    def obtener_nombre_servicio(self) -> str:
        return "Criogenia con Nitrógeno Líquido (-196°C)"


print("--- Probando Facturador Modular (Cumpliendo OCP) ---\n")
facturador = FacturadorBodega("Frigoríficos del Norte S.A.S.", descuento_por_volumen=0.10)

# Servicios tradicionales
facturador.imprimir_cuenta("AgroFrutas S.A.", TarifaRefrigerado(), pallets=30, dias=15)
facturador.imprimir_cuenta("Avícola Central", TarifaCongelado(), pallets=60, dias=30)
facturador.imprimir_cuenta("Laboratorio Farma", TarifaUltraCongelado(), pallets=10, dias=20)

# NUEVO SERVICIO FUNCIONANDO SIN HABER TOCADO EL FACTURADOR
facturador.imprimir_cuenta("BioBanco Células Madre", TarifaCriogenica(), pallets=5, dias=30)


--- Probando Facturador Modular (Cumpliendo OCP) ---

Factura: Frigoríficos del Norte S.A.S.
Cliente: AgroFrutas S.A.
Tipo: Refrigeración Frescos (+4°C con control de humedad)
Cantidad: 30 pallets por 15 días
Subtotal: $2,475,000.00 | Descuento: $0.00 | Total: $2,475,000.00 COP
----------------------------------------------------------------------
Factura: Frigoríficos del Norte S.A.S.
Cliente: Avícola Central
Tipo: Congelados Estándar (-18°C)
Cantidad: 60 pallets por 30 días
Subtotal: $28,080,000.00 | Descuento: $2,808,000.00 | Total: $25,272,000.00 COP
----------------------------------------------------------------------
Factura: Frigoríficos del Norte S.A.S.
Cliente: Laboratorio Farma
Tipo: Ultra-Freezer Farmacéutico (-80°C con generador de respaldo)
Cantidad: 10 pallets por 20 días
Subtotal: $10,700,000.00 | Descuento: $0.00 | Total: $10,700,000.00 COP
----------------------------------------------------------------------
Factura: Frigoríficos del Norte S.A.S.
Cliente: BioBanco Cé